In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import gc
import re
import math
import json
import time
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

import kagglehub

warnings.filterwarnings("ignore")

# -------------------------
# Load models
# -------------------------
qwen_model_path = kagglehub.model_download("qwen-lm/qwen2.5-math/transformers/7b")
deepseek_model_path = kagglehub.model_download("deepseek-ai/deepseek-math/pytorch/deepseek-math-7b-instruct")
qwen_model_path2 = kagglehub.model_download("qwen-lm/qwen-3/transformers/1.7b")

In [ ]:
# -------------------------
# Basic configuration
# -------------------------
ANSWER_MOD = 100000
DEBUG = True
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -------------------------
# Competition paths
# -------------------------
COMPETITION_DIR_CANDIDATES = [
    "/kaggle/input/ai-mathematical-olympiad-progress-prize-3",
    "/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3",
]

MODEL_PATHS = {
    "qwen_math_7b": qwen_model_path,
    "deepseek_math_7b": deepseek_model_path,
    "qwen_3": qwen_model_path2,
}

# -------------------------
# Sampling plan
# Adjust these later after timing tests
# -------------------------
MODEL_CONFIGS = {
    "qwen_math_7b": {
        "enabled": False,
        "n_samples": 4,
        "temperature": 0.7,
        "top_p": 0.95,
        "max_new_tokens": 768,
        "weight": 1.25,
    },
    "deepseek_math_7b": {
        "enabled": False,
        "n_samples": 4,
        "temperature": 0.7,
        "top_p": 0.95,
        "max_new_tokens": 768,
        "weight": 1.10,
    },
    "qwen_3": {
        "enabled": True,
        "n_samples": 4,
        "temperature": 0.7,
        "top_p": 0.95,
        "max_new_tokens": 768,
        "weight": 1.50,
    },
}

In [ ]:
# -------------------------
# Utilities
# -------------------------
def set_debug(*args):
    if DEBUG:
        print(*args)

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def resolve_competition_dir():
    for path in COMPETITION_DIR_CANDIDATES:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        "Competition input directory was not found. "
        "Please check the actual path under /kaggle/input."
    )

competition_dir = resolve_competition_dir()
test_path = os.path.join(competition_dir, "test.csv")
sample_submission_path = os.path.join(competition_dir, "sample_submission.csv")

test_df = pd.read_csv(test_path)

set_debug("Competition directory:", competition_dir)
set_debug("test.csv shape:", test_df.shape)
set_debug(test_df.head())

In [ ]:
# -------------------------
# Detect problem column
# -------------------------
problem_column = None
for candidate in ["problem", "question", "prompt"]:
    if candidate in test_df.columns:
        problem_column = candidate
        break

if problem_column is None:
    raise ValueError(
        "No problem text column was found. "
        "Expected one of: problem, question, prompt"
    )

In [ ]:
# -------------------------
# Answer extraction
# -------------------------
def normalize_answer_int(x):
    try:
        return int(x) % ANSWER_MOD
    except Exception:
        return 0

def extract_boxed_number(text: str):
    if not isinstance(text, str):
        return None

    patterns = [
        r'\\boxed\{(-?\d+)\}',
        r'boxed\{(-?\d+)\}',
        r'\\boxed\s*\{?\s*(-?\d+)\s*\}?',
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return normalize_answer_int(match.group(1))
    return None

def extract_last_integer(text: str):
    if not isinstance(text, str):
        return None
    matches = re.findall(r'-?\d+', text)
    if matches:
        return normalize_answer_int(matches[-1])
    return None

def extract_answer(text: str):
    boxed = extract_boxed_number(text)
    if boxed is not None:
        return boxed
    last_int = extract_last_integer(text)
    if last_int is not None:
        return last_int
    return 0

In [ ]:
# -------------------------
# Prompt templates
# -------------------------
def make_solver_prompt(problem: str, model_name: str) -> str:
    return f"""
You are an expert mathematical reasoning assistant.

Solve the following competition mathematics problem carefully.
Reason step by step.
Prefer exact symbolic reasoning over vague intuition.
If useful, mentally simulate algebraic or arithmetic verification.
At the end, give only one final integer answer modulo 100000 if needed.

Final answer format:
\\boxed{{answer}}

Problem:
{problem}

Solution:
""".strip()

def make_verifier_prompt(problem: str, candidate_answer: int) -> str:
    return f"""
You are verifying a mathematical answer.

Check whether the candidate answer is plausible for the following competition math problem.
Reason carefully and independently.
If the candidate answer is wrong, derive the correct final answer.
Return exactly one final integer answer in the format \\boxed{{answer}}.

Problem:
{problem}

Candidate answer:
{candidate_answer}

Verification:
""".strip()


In [ ]:
# -------------------------
# Model loading (H100 optimized)
# -------------------------
loaded_models = {}
loaded_tokenizers = {}
available_model_names = []

def load_single_model(model_name: str, model_path: str):
    if not os.path.exists(model_path):
        set_debug(f"[Skip] Model path not found: {model_name} -> {model_path}")
        return False

    try:
        set_debug(f"[Load] {model_name} from {model_path}")

        tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            trust_remote_code=True,
        )

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        # H100 optimization: bfloat16 for better performance
        dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
        
        # Check GPU memory and set device_map appropriately
        if torch.cuda.is_available():
            gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3  # GB
            if gpu_mem >= 80:  # H100/A100 80GB
                device_map = "auto"  # Can fit all models on single H100
            else:
                device_map = {"": 0}  # Single GPU for smaller cards
        else:
            device_map = "cpu"

        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            trust_remote_code=True,
            dtype=dtype,
            device_map=device_map,
            low_cpu_mem_usage=True,
        )
        model.eval()

        loaded_models[model_name] = model
        loaded_tokenizers[model_name] = tokenizer
        available_model_names.append(model_name)

        set_debug(f"[OK] Loaded {model_name}")
        return True

    except Exception as e:
        set_debug(f"[Fail] Could not load {model_name}: {repr(e)}")
        cleanup_memory()
        return False

for model_name, model_path in MODEL_PATHS.items():
    if MODEL_CONFIGS[model_name]["enabled"]:
        load_single_model(model_name, model_path)

if len(available_model_names) == 0:
    raise RuntimeError("No model could be loaded. Please check the model paths and GPU memory.")

set_debug("Available models:", available_model_names)

In [ ]:
# -------------------------
# Generation (H100 optimized)
# -------------------------
def generate_once(model_name: str, prompt: str):
    model = loaded_models[model_name]
    tokenizer = loaded_tokenizers[model_name]
    cfg = MODEL_CONFIGS[model_name]

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    )

    model_device = next(model.parameters()).device
    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask", None),
            max_new_tokens=cfg["max_new_tokens"],
            do_sample=True,
            temperature=cfg["temperature"],
            top_p=cfg["top_p"],
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05,
            use_cache=True,  # H100 optimization
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if text.startswith(prompt):
        text = text[len(prompt):].strip()

    return text

def solve_with_single_model(problem: str, model_name: str):
    cfg = MODEL_CONFIGS[model_name]
    answers = []
    raw_outputs = []

    for _ in range(cfg["n_samples"]):
        prompt = make_solver_prompt(problem, model_name)
        raw_text = generate_once(model_name, prompt)
        ans = extract_answer(raw_text)
        answers.append(ans)
        raw_outputs.append(raw_text)

    counter = Counter(answers)
    best_answer, best_count = counter.most_common(1)[0]

    return {
        "model_name": model_name,
        "answers": answers,
        "raw_outputs": raw_outputs,
        "best_answer": normalize_answer_int(best_answer),
        "best_count": best_count,
        "counter": counter,
    }


In [ ]:
# -------------------------
# Weighted voting
# -------------------------
def weighted_vote(model_results):
    score_map = {}

    for result in model_results:
        model_name = result["model_name"]
        model_weight = MODEL_CONFIGS[model_name]["weight"]

        local_counter = result["counter"]
        total = sum(local_counter.values())

        for ans, cnt in local_counter.items():
            confidence = cnt / max(total, 1)
            score = model_weight * confidence
            score_map[ans] = score_map.get(ans, 0.0) + score

    best_answer = max(score_map.items(), key=lambda x: x[1])[0]
    return normalize_answer_int(best_answer), score_map

# -------------------------
# Optional verification pass
# -------------------------
def verify_with_secondary_model(problem: str, candidate_answer: int, verifier_model_name: str):
    prompt = make_verifier_prompt(problem, candidate_answer)
    raw_text = generate_once(verifier_model_name, prompt)
    verified_answer = extract_answer(raw_text)
    return verified_answer, raw_text

In [ ]:
# -------------------------
# Final prediction
# -------------------------
def predict_answer(problem: str):
    model_results = []

    for model_name in available_model_names:
        try:
            result = solve_with_single_model(problem, model_name)
            model_results.append(result)
        except Exception as e:
            set_debug(f"[Warn] Inference failed for {model_name}: {repr(e)}")
            cleanup_memory()

    if len(model_results) == 0:
        return 0

    candidate_answer, score_map = weighted_vote(model_results)

    # ここを追加して、`score_map` を確認する
    print("score_map:", score_map)

    # Optional lightweight verification:
    # If DeepSeek is available, verify the voted answer once.
    if "deepseek_math_7b" in available_model_names:
        try:
            verified_answer, _ = verify_with_secondary_model(
                problem=problem,
                candidate_answer=candidate_answer,
                verifier_model_name="deepseek_math_7b",
            )
            # Use verifier answer only if it agrees with one of the model-major answers.
            major_answers = [r["best_answer"] for r in model_results]
            if verified_answer in major_answers:
                candidate_answer = verified_answer
        except Exception as e:
            set_debug(f"[Warn] Verification failed: {repr(e)}")
            cleanup_memory()

    return normalize_answer_int(candidate_answer)

In [ ]:
# -------------------------
# Run inference
# -------------------------
submission_df = pd.DataFrame()
submission_df["id"] = test_df["id"]

predictions = []
start_time = time.time()

for idx, row in test_df.iterrows():
    problem_text = row[problem_column]

    try:
        pred = predict_answer(problem_text)
    except Exception as e:
        set_debug(f"[Error] Failed on row {idx}: {repr(e)}")
        pred = 0
        cleanup_memory()

    predictions.append(pred)

    if DEBUG:
        elapsed = time.time() - start_time
        set_debug(f"[{idx+1}/{len(test_df)}] pred={pred} elapsed={elapsed:.1f}s")

submission_df["answer"] = np.array(predictions).astype(int) % ANSWER_MOD
submission_df["answer"] = submission_df["answer"].clip(0, ANSWER_MOD - 1)

output_path = "/kaggle/working/submission.csv"
submission_df.to_csv(output_path, index=False)

print("Saved submission file to:", output_path)
print(submission_df.head())
print("Available models actually used:", available_model_names)